# PIPS — benchmark tests notebook

Runs the same four benchmark modes as `pips-benchmark`, one cell at a time, and shows the results inline.

| Section | Costs API credits? |
|---|---|
| 1. Routing check, all 4 modes | **No.** Keyword routing and the cross-encoder run locally. LLM keywords come from the cached `data/generated_keywords.json`. |
| 2. Full benchmark per mode | **Yes.** 40 Anthropic calls per mode. |
| 3. Comparison table | No. It reads `results/*.json`. |
| 4. Answer inspector | No. It reads `results/*.json`. |

Use the **PIPS (.venv)** kernel. Outputs below are from the last run, so you can read results without re-running anything.

In [1]:
# ── Config ───────────────────────────────────
# Set a mode to True to run its full (paid) benchmark in Section 2.
RUN_MODES = {
    "manual":    False,
    "llm":       False,
    "cross":     False,
    "llm_cross": False,
}
REBUILD_LLM_INDEX = False   # True = regenerate data/generated_keywords.json (makes API calls)
CANDIDATE_K = 5             # cross-encoder candidate pool, same default as pips-benchmark

In [2]:
import os, sys, json, html
from pathlib import Path

# Work from the project root (one level up from notebooks/) so .env, data/ and results/ resolve.
ROOT = Path.cwd()
if not (ROOT / "src" / "pips").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "pips").is_dir():
    raise RuntimeError(f"Can't find the PIPS project root from {Path.cwd()}")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))   # works even if the package isn't pip-installed

from IPython.display import HTML, display

from pips.prompts.enterprise_assistant import PROMPT_SECTIONS, FULL_SYSTEM_PROMPT, SLICED_BASE_PROMPT
from pips.routing.llm_keyword_index import apply_keywords, load_or_generate_llm_keywords
from pips.evaluation.benchmark import MODEL_NAME, TEST_QUERIES, count_tokens
from pips.cli import benchmark as runner

print("Model           :", MODEL_NAME)
print("API key loaded  :", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("Sections        :", len(PROMPT_SECTIONS))
print("Test queries    :", len(TEST_QUERIES))
print("Full prompt est.:", count_tokens(FULL_SYSTEM_PROMPT), "tokens (chars/4)")

Model           : claude-sonnet-4-6
API key loaded  : True
Sections        : 15
Test queries    : 10
Full prompt est.: 33798 tokens (chars/4)


In [3]:
# ── Display helpers ──────────────────────────
def table(headers, rows):
    th = "".join(f"<th style='text-align:left'>{html.escape(str(h))}</th>" for h in headers)
    trs = "".join(
        "<tr>" + "".join(f"<td style='text-align:left'>{c}</td>" for c in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{th}</tr></thead><tbody>{trs}</tbody></table>"))

def mark(ok):
    return "<span style='color:#1a7f37'>✓ PASS</span>" if ok else "<span style='color:#cf222e'>✗ FAIL</span>"

def show_results(results):
    rows = [[
        r["id"], html.escape(r["query"]), r["expected_section"],
        ", ".join(r["matched_sections"]), mark(r["routing_correct"]),
        r["full"]["tokens"], r["sliced"]["tokens"], f"{r['token_reduction_pct']:.1f}%",
        f"{r['full']['score']}/10", f"{r['sliced']['score']}/10",
    ] for r in results]
    table(["ID", "Query", "Expected", "Selected", "Routing", "Full tok", "Sliced tok",
           "Reduction", "Full score", "Sliced score"], rows)
    s = runner.summarize_results("", results)
    print(f"Routing {s['routing_acc']:.0f}% | avg reduction {s['avg_reduction']:.1f}% | "
          f"avg sliced score {s['sliced_score']:.1f}/10 over {s['queries']} queries")

## 1. Routing check, all 4 modes (free)

For each query, this shows which 2 sections each mode would put in the sliced prompt, and whether the expected section is among them. It makes no Anthropic calls, as long as `data/generated_keywords.json` exists and `REBUILD_LLM_INDEX` is False.

In [4]:
manual_sections = PROMPT_SECTIONS
llm_sections = apply_keywords(
    PROMPT_SECTIONS,
    load_or_generate_llm_keywords(PROMPT_SECTIONS, model_name=MODEL_NAME, rebuild=REBUILD_LLM_INDEX),
)
manual_tree = runner.build_tree(manual_sections)
llm_tree = runner.build_tree(llm_sections)
print(manual_tree.describe())

PageIndexTree (15 nodes)
  [billing_payments]  keywords=18
  [technical_support]  keywords=20
  [account_management]  keywords=20
  [product_features]  keywords=21
  [returns_refunds]  keywords=13
  [security_privacy]  keywords=18
  [hr_policies]  keywords=20
  [sales_pricing]  keywords=18
  [onboarding]  keywords=16
  [legal_compliance]  keywords=17
  [customer_success]  keywords=14
  [infrastructure_sla]  keywords=15
  [partner_program]  keywords=13
  [accessibility_localization]  keywords=14
  [ai_features]  keywords=14


In [5]:
# Loads the local cross-encoder (downloads it from Hugging Face on first run).
from pips.routing.cross_encoder_reranker import CrossEncoderReranker
reranker = CrossEncoderReranker(os.environ.get("CROSS_ENCODER_MODEL", "cross-encoder/ms-marco-MiniLM-L6-v2"))
print("Reranker:", reranker.model_name)

/home/ranga/PIPS/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4743.21it/s]

Reranker: cross-encoder/ms-marco-MiniLM-L6-v2


In [6]:
ROUTERS = {
    "manual":    dict(tree=manual_tree, reranker=None,            candidate_k=None),
    "llm":       dict(tree=llm_tree,    reranker=None,            candidate_k=None),
    "cross":     dict(tree=manual_tree, reranker=reranker.rerank, candidate_k=CANDIDATE_K),
    "llm_cross": dict(tree=llm_tree,    reranker=reranker.rerank, candidate_k=CANDIDATE_K),
}

routing_hits = {m: 0 for m in ROUTERS}
rows = []
for t in TEST_QUERIES:
    row = [t["id"], html.escape(t["query"]), f"<b>{t['expected_section']}</b>"]
    for mode, r in ROUTERS.items():
        _, ids = r["tree"].slice(t["query"], SLICED_BASE_PROMPT, top_k=2,
                                 reranker=r["reranker"], candidate_k=r["candidate_k"])
        ok = t["expected_section"] in ids
        routing_hits[mode] += ok
        row.append(f"{mark(ok)}<br><small>{', '.join(ids)}</small>")
    rows.append(row)

table(["ID", "Query", "Expected"] + list(ROUTERS), rows)
n = len(TEST_QUERIES)
for mode, hits in routing_hits.items():
    print(f"{mode:<10} routing accuracy: {hits}/{n} ({hits / n * 100:.0f}%)")

ID,Query,Expected,manual,llm,cross,llm_cross
Q01,My invoice shows a wrong charge and I want a refund for the billing error.,billing_payments,"✓ PASSbilling_payments, technical_support","✓ PASSbilling_payments, returns_refunds","✓ PASSreturns_refunds, billing_payments","✓ PASSreturns_refunds, billing_payments"
Q02,I am getting ERR_API_429 error when calling your REST API.,technical_support,"✓ PASStechnical_support, onboarding","✓ PASStechnical_support, onboarding","✓ PASStechnical_support, onboarding","✓ PASStechnical_support, accessibility_localization"
Q03,How do I set up SSO with Okta for my enterprise account?,account_management,"✓ PASSaccount_management, onboarding","✓ PASSsales_pricing, account_management","✓ PASSonboarding, account_management","✓ PASSonboarding, account_management"
Q04,Does the Professional plan include Gantt chart and time tracking?,product_features,"✓ PASSproduct_features, billing_payments","✓ PASSproduct_features, hr_policies","✓ PASSproduct_features, onboarding","✓ PASSproduct_features, billing_payments"
Q05,I want to cancel my subscription and get my money back.,returns_refunds,"✓ PASSreturns_refunds, billing_payments","✓ PASSreturns_refunds, hr_policies","✓ PASSreturns_refunds, billing_payments","✓ PASSreturns_refunds, billing_payments"
Q06,Is your platform HIPAA compliant? We store patient medical data.,security_privacy,"✓ PASSsecurity_privacy, legal_compliance","✓ PASSsecurity_privacy, legal_compliance","✓ PASSsecurity_privacy, legal_compliance","✓ PASSsecurity_privacy, legal_compliance"
Q07,What is the parental leave policy and how many weeks are covered?,hr_policies,"✓ PASShr_policies, onboarding","✓ PASShr_policies, legal_compliance","✓ PASShr_policies, legal_compliance","✓ PASShr_policies, returns_refunds"
Q08,We have 300 users. Is there a volume discount available for us?,sales_pricing,"✓ PASSsales_pricing, ai_features","✓ PASSsales_pricing, billing_payments","✓ PASSsales_pricing, ai_features","✓ PASSsales_pricing, billing_payments"
Q09,How do I import my existing boards and projects from Trello?,onboarding,"✓ PASSproduct_features, onboarding","✓ PASSonboarding, account_management","✓ PASSonboarding, product_features","✓ PASSonboarding, account_management"
Q10,Who owns the data I store in your platform? Does your company have any rights over it?,legal_compliance,"✓ PASSsecurity_privacy, legal_compliance","✓ PASSsecurity_privacy, legal_compliance","✓ PASSlegal_compliance, security_privacy","✓ PASSlegal_compliance, security_privacy"


manual     routing accuracy: 10/10 (100%)
llm        routing accuracy: 10/10 (100%)
cross      routing accuracy: 10/10 (100%)
llm_cross  routing accuracy: 10/10 (100%)


## 2. Full benchmark per mode (paid: 40 API calls each)

Each cell runs only when its flag in `RUN_MODES` is True. It writes the same files as `pips-benchmark --mode all`:
`results/benchmark_results_<mode>.json` and `results/report_<mode>.txt`, with `_partial` checkpoints after each query.

### Mode: `manual` (manual keywords)

In [7]:
if RUN_MODES["manual"]:
    runner.run_case(label="manual keywords", sections=manual_sections, full_prompt=FULL_SYSTEM_PROMPT,
                  suffix="manual", )
    show_results(json.loads(Path("results/benchmark_results_manual.json").read_text()))
else:
    print('Skipped. Set RUN_MODES["manual"] = True to run it.')

Skipped. Set RUN_MODES["manual"] = True to run it.


### Mode: `llm` (LLM-generated keywords)

In [8]:
if RUN_MODES["llm"]:
    runner.run_case(label="LLM-generated keywords", sections=llm_sections, full_prompt=FULL_SYSTEM_PROMPT,
                  suffix="llm", )
    show_results(json.loads(Path("results/benchmark_results_llm.json").read_text()))
else:
    print('Skipped. Set RUN_MODES["llm"] = True to run it.')

Skipped. Set RUN_MODES["llm"] = True to run it.


### Mode: `cross` (manual keywords + cross-encoder rerank)

In [9]:
if RUN_MODES["cross"]:
    runner.run_case(label="manual keywords + cross-encoder rerank", sections=manual_sections, full_prompt=FULL_SYSTEM_PROMPT,
                  suffix="cross", reranker=reranker, candidate_k=CANDIDATE_K,)
    show_results(json.loads(Path("results/benchmark_results_cross.json").read_text()))
else:
    print('Skipped. Set RUN_MODES["cross"] = True to run it.')

Skipped. Set RUN_MODES["cross"] = True to run it.


### Mode: `llm_cross` (LLM keywords + cross-encoder rerank)

In [10]:
if RUN_MODES["llm_cross"]:
    runner.run_case(label="LLM keywords + cross-encoder rerank", sections=llm_sections, full_prompt=FULL_SYSTEM_PROMPT,
                  suffix="llm_cross", reranker=reranker, candidate_k=CANDIDATE_K,)
    show_results(json.loads(Path("results/benchmark_results_llm_cross.json").read_text()))
else:
    print('Skipped. Set RUN_MODES["llm_cross"] = True to run it.')

Skipped. Set RUN_MODES["llm_cross"] = True to run it.


## 3. Comparison across modes (free)

Reads whichever `results/benchmark_results_<mode>.json` files exist, including results from earlier runs.

In [11]:
RESULT_FILES = {m: Path(f"results/benchmark_results_{m}.json") for m in ROUTERS}
loaded = {m: json.loads(p.read_text()) for m, p in RESULT_FILES.items() if p.exists()}

rows = []
for mode, res in loaded.items():
    s = runner.summarize_results(mode, res)
    full_score = sum(r["full"]["score"] for r in res) / max(1, len(res))
    rows.append([mode, s["queries"], f"{s['avg_full_tokens']:.0f}", f"{s['avg_sliced_tokens']:.0f}",
                 f"{s['avg_reduction']:.1f}%", f"{s['routing_acc']:.0f}%",
                 f"{full_score:.1f}", f"{s['sliced_score']:.1f}", f"{s['sliced_score'] - full_score:+.1f}"])
table(["Mode", "Rows", "Full tok", "Sliced tok", "Reduction", "Routing", "Full score", "Sliced score", "Delta"], rows)
missing = [m for m in RESULT_FILES if m not in loaded]
if missing:
    print("No results yet for:", ", ".join(missing))

Mode,Rows,Full tok,Sliced tok,Reduction,Routing,Full score,Sliced score,Delta
manual,10,30105,4221,86.0%,100%,8.2,8.2,+0.0
llm,10,30105,4197,86.1%,100%,8.1,8.2,+0.1
cross,10,30105,4160,86.2%,100%,8.1,7.9,-0.2
llm_cross,10,30105,4247,85.9%,100%,8.3,8.0,-0.3


In [12]:
# Same as `pips-compare`: also writes results/comparison_summary.txt
from pips.cli import compare
compare.main()

|══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════|
|     PAGEINDEX PROMPT SLICING — CONSOLIDATED COMPARISON                                                               |
|══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════|
|                                                                                                                      |
|  Mode                               Rows  Full Tok   Sliced Tok  Reduc%   Routing  Full     Sliced   Delta           |
|──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────|
|  Manual keywords                    10    30105      4221        86.0%    100%     8.2     8.2     +0.0              |
|  LLM-generated keywords             10    30105      4197        86.1%    100%     8.1     8.2     +0.1              |
|  Manual keywords + cross-encod

## 4. Answer inspector (free)

Shows the full-prompt and sliced-prompt answers for one mode side by side, with the judge's reasoning for each.

In [13]:
INSPECT_MODE = "manual"   # manual | llm | cross | llm_cross
ONLY_FAILURES = False     # True = only rows with wrong routing or sliced score < full score

res = loaded.get(INSPECT_MODE) or []
if not res:
    print(f"No results for {INSPECT_MODE}")
show_results(res) if res else None
for r in res:
    if ONLY_FAILURES and r["routing_correct"] and r["sliced"]["score"] >= r["full"]["score"]:
        continue
    def cell(side):
        d = r[side]
        return (f"<b>{side}</b>: {d['score']}/10, {d['tokens']} tok, {d['latency_s']:.1f}s<br>"
                f"<pre style='white-space:pre-wrap'>{html.escape(d['answer'])}</pre>"
                f"<i>Judge: {html.escape(str(d['reasoning']))}</i>")
    display(HTML(f"<h4>{r['id']}: {html.escape(r['query'])}</h4>"))
    table(["Full prompt", "Sliced prompt"], [[cell("full"), cell("sliced")]])

ID,Query,Expected,Selected,Routing,Full tok,Sliced tok,Reduction,Full score,Sliced score
Q01,My invoice shows a wrong charge and I want a refund for the billing error.,billing_payments,"billing_payments, technical_support",✓ PASS,30107,4634,84.6%,9/10,9/10
Q02,I am getting ERR_API_429 error when calling your REST API.,technical_support,"technical_support, onboarding",✓ PASS,30107,4283,85.8%,8/10,8/10
Q03,How do I set up SSO with Okta for my enterprise account?,account_management,"account_management, onboarding",✓ PASS,30106,4186,86.1%,9/10,9/10
Q04,Does the Professional plan include Gantt chart and time tracking?,product_features,"product_features, billing_payments",✓ PASS,30103,4452,85.2%,9/10,9/10
Q05,I want to cancel my subscription and get my money back.,returns_refunds,"returns_refunds, billing_payments",✓ PASS,30102,4452,85.2%,6/10,5/10
Q06,Is your platform HIPAA compliant? We store patient medical data.,security_privacy,"security_privacy, legal_compliance",✓ PASS,30105,4062,86.5%,7/10,7/10
Q07,What is the parental leave policy and how many weeks are covered?,hr_policies,"hr_policies, onboarding",✓ PASS,30104,4152,86.2%,10/10,10/10
Q08,We have 300 users. Is there a volume discount available for us?,sales_pricing,"sales_pricing, ai_features",✓ PASS,30106,3820,87.3%,8/10,8/10
Q09,How do I import my existing boards and projects from Trello?,onboarding,"product_features, onboarding",✓ PASS,30104,4102,86.4%,8/10,8/10
Q10,Who owns the data I store in your platform? Does your company have any rights over it?,legal_compliance,"security_privacy, legal_compliance",✓ PASS,30109,4066,86.5%,8/10,9/10


Routing 100% | avg reduction 86.0% | avg sliced score 8.2/10 over 10 queries


Full prompt,Sliced prompt
"full: 9/10, 30107 tok, 10.5sI'd be happy to help you resolve this billing error. Here's how we'll approach it: ## Immediate Next Steps To investigate and correct the charge, I'll need a few details from you: 1. **Invoice number** and the date of the charge in question 2. **Description of the error** — for example, wrong plan charged, duplicate charge, incorrect quantity, unexpected fee, etc. 3. **Your role on the account** — billing corrections require an Owner or billing admin to authorize changes ## What to Expect Once I can verify the details: - **Verified billing errors** are eligible for a **full refund**, processed within **5 business days** - If a duplicate charge is involved, I'll confirm whether both transactions fully settled (vs. one being an authorization hold) before initiating the refund - The dispute must be raised **within 60 days of the charge** — if you're close to or past that window, please let me know and I can note any extenuating circumstances ## How to Submit Formally You can also raise a billing dispute directly via: - **Support ticket** (this conversation qualifies) - **Email:** billing@company.com ## A Note on Chargebacks If you're considering filing a chargeback with your bank, please reach out to us first — we can typically resolve verified billing errors much faster through our process, and chargebacks result in account suspension pending investigation. --- Please share the invoice details and a description of what looks incorrect, and I'll open the review right away.Judge: The AI answer correctly captures all key ground-truth facts: the 60-day dispute window, the two submission methods (support ticket and billing@company.com), and the resolution timeframe (5-7 business days, though the AI says '5 business days' which is slightly imprecise but within range).","sliced: 9/10, 4634 tok, 9.8sI'd be happy to help you with your billing dispute and potential refund. To make sure I handle this accurately, I'll need a few details from you. ## Information Needed Please provide the following so I can review your case: 1. **Invoice number** and the date of the charge in question 2. **Your account's billing role** (Owner or Billing Admin — I'll need to verify your authority before discussing invoice details) 3. **The nature of the error** — for example: - Incorrect plan charged - Duplicate charge - Wrong number of seats/users billed - Unexpected fee (e.g., late fee, tax) - Other discrepancy --- ## Important Things to Know - **Dispute window:** Billing disputes must be raised **within 60 days** of the charge. If your charge is outside this window, I can still look into verified platform errors, but exceptions may require escalation to our Billing Director. - **Resolution time:** Once submitted, disputes are typically resolved within **5–7 business days**. - **How to submit formally:** You can also raise a dispute via a support ticket or by emailing **billing@company.com**. --- ## Next Steps Once you share the details above, I can: - Review the charge against your plan and billing history - Determine whether this is a **ledger/platform error** (eligible for correction) or a **policy-based request** (which may follow a different process) - Escalate to the appropriate team if needed — disputes **over $500** follow an expedited escalation path involving our Billing Director Please go ahead and share those details, and I'll get this moving for you right away.Judge: The AI answer accurately covers all three ground-truth facts: the 60-day dispute window, the submission methods (support ticket or billing@company.com), and the 5-7 business day resolution time."


Full prompt,Sliced prompt
"full: 8/10, 30107 tok, 9.8s## ERR_API_429 — Rate Limit Exceeded This error means your application has exceeded the number of API requests allowed per minute for your current plan. Here's how to resolve it: --- ### Your Plan's Rate Limits | Plan | Requests per Minute | |---|---| | Starter | 100 | | Professional | 500 | | Enterprise | 2,000 | --- ### Immediate Steps 1. **Wait 60 seconds** before retrying — the rate limit window resets per minute. 2. **Check your current plan** to confirm which limit applies to your account. 3. **Review your API call patterns** to identify if there are redundant or overly frequent requests. --- ### Longer-Term Fixes - **Implement exponential backoff** — when you receive a 429, wait progressively longer between retries rather than retrying immediately. - **Use idempotency keys** to avoid duplicate requests if you're retrying failed calls. - **Batch your requests** where possible to reduce total call volume. - **Cache responses** for data that doesn't change frequently to avoid repeated identical calls. --- ### If You Need Higher Limits - **Enterprise customers** can request a **temporary import limit review** if you have a time-sensitive, high-volume use case (e.g., a large data import). Please provide your expected volume and deadline. - If you're on Starter or Professional and consistently hitting limits, upgrading your plan would increase your rate allowance. --- ### Useful Reference - Your API keys are managed under **Settings > Developer > API Keys** - Base API URL: `https://api.company.com/v2/` --- Could you share a bit more context — such as yourJudge: The AI answer correctly identifies ERR_API_429 as rate limit exceeded, advises checking plan limits, and explicitly states to wait 60 seconds before retrying, all matching the ground-truth facts, though it adds extra details not in the ground truth.","sliced: 8/10, 4283 tok, 8.4s## ERR_API_429 – Rate Limit Exceeded You're hitting the API rate limit for your current plan. Here's how to resolve it: --- ### Immediate Steps 1. **Wait 60 seconds** before retrying your request — the rate limit window resets automatically. 2. **Check your current plan limits:** | Plan | Rate Limit | |------|-----------| | Starter | 100 requests/min | | Pro | 500 requests/min | | Enterprise | 2,000 requests/min | --- ### Longer-Term Recommendations - **Chunk large operations** – Break bulk requests (e.g., large imports) into smaller batches that stay within your plan's per-minute limit. - **Implement exponential backoff** – After receiving a 429, wait progressively longer between retries rather than immediately retrying. - **Optimize request frequency** – Review whether any automated processes are making redundant or unnecessary API calls. --- ### If You're on a Bulk Import If you're trying to import a large number of records (e.g., 10,000+), chunking and exponential backoff are especially important. **Enterprise customers** can also request a **temporary import limit review** — please share your expected volume and deadline so I can open that review. --- ### Next Steps To help you further, could you share: - Your **current plan** (Starter, Pro, or Enterprise)? - The **API endpoint** you're calling? - Whether this is happening during a **bulk operation or regular usage**? This will help determine if your use case warrants a plan upgrade or a temporary limit exception.Judge: The AI answer correctly identifies ERR_API_429 as rate limit exceeded, advises checking plan limits, and explicitly states to wait 60 seconds before retrying, covering all three ground-truth facts, though it also adds extra information not in the ground truth."


Full prompt,Sliced prompt
"full: 9/10, 30106 tok, 9.1s# Setting Up SSO with Okta for Your Enterprise Account SSO via SAML 2.0 is available on the **Enterprise plan**. Here's a step-by-step guide to get you configured: --- ## Required Okta Field Mappings Before you begin, ensure your Okta app is configured with the following SAML attributes: | Field | Value | |-------|-------| | **NameID** | User's primary email address (email format) | | **firstName** | `given_name` | | **lastName** | `family_name` | | **email** | `email` | > ⚠️ The NameID **must** be in email format — a common misconfiguration that can block logins. --- ## Setup Steps 1. **Create a SAML 2.0 app** in your Okta Admin Console 2. **Configure the attribute mappings** as listed above 3. **Copy from Okta:** - IdP SSO URL - X.509 signing certificate 4. **Paste into TechCorp:** Navigate to **Settings > SSO** and enter the IdP SSO URL and certificate 5. **Test with enforcement OFF** — sign in with a test user before enforcing SSO org-wide 6. **Enable enforcement** once you've confirmed successful logins --- ## Additional Capabilities Available on Enterprise - **Just-in-Time (JIT) provisioning** — automatically creates user accounts on first SSO login - **SCIM provisioning** — automates the full user lifecycle (create, update, deactivate) synced from Okta --- ## Important Cautions - **Don't enforce SSO until testing is complete**Judge: The AI answer correctly identifies SAML 2.0, Enterprise plan requirement, Okta support, and SCIM provisioning, all of which are the key ground-truth facts.","sliced: 9/10, 4186 tok, 9.0s# Setting Up SSO with Okta for Enterprise Here's a step-by-step guide to get your Okta SSO configured correctly. ## Prerequisites - You must be on an **Enterprise plan** (SSO/SAML 2.0 is Enterprise-only) - You need **Owner or Admin** role to access SSO settings --- ## Configuration Steps ### 1. Gather Required Okta Settings Ensure the following fields are correctly mapped in your Okta SAML application: | Field | Required Value | |---|---| | **NameID** | User's primary email (email format) | | **firstName** | mapped to `given_name` | | **lastName** | mapped to `family_name` | | **email** | mapped to `email` | > ⚠️ **Important:** NameID *must* be in email format. A mismatch here is a common cause of SSO failures. ### 2. Configure in TechCorp Platform 1. Go to **Settings > SSO** 2. Select **SAML 2.0** as the protocol 3. Paste in your **IdP SSO URL** from Okta 4. Upload or paste your **IdP certificate** from Okta ### 3. Test Before Enforcing - **Keep SSO enforcement turned OFF** during initial setup - Test login with one or two users first - Only enable enforcement after successful validation --- ## Optional Enhancements (Enterprise) - **JIT Provisioning:** Automatically creates user accounts on first SSO login - **SCIM Provisioning:** Enables automated user lifecycle management (adds/removes users based on Okta directory changes) --- ## Important CaJudge: The answer correctly identifies SAML 2.0, Enterprise plan requirement, Okta support, and SCIM provisioning, all matching the ground truth facts."


...

Full prompt,Sliced prompt
"full: 7/10, 30105 tok, 9.8s## HIPAA Compliance on TechCorp Platform Thank you for asking before storing patient data — that's exactly the right approach. ### Short Answer Yes, HIPAA compliance is available on our platform, but **specific requirements must be met before you can store any Protected Health Information (PHI)**. --- ### Requirements to Store PHI All three of the following must be in place **before** using the platform for PHI workflows: 1. **Enterprise Plan** — HIPAA compliance is an Enterprise-only capability 2. **HIPAA Module** — Available as an add-on at **$200/month**, must be explicitly enabled 3. **Signed Business Associate Agreement (BAA)** — Required contractual agreement between your organization and TechCorp > ⚠️ **Important:** Do not store patient medical data on the platform until all three of these are confirmed and active. --- ### What the HIPAA Module Includes Once properly configured, the Enterprise + HIPAA setup provides: - **SOC 2 Type II** certified infrastructure - **AES-256 encryption** at rest and **TLS 1.3** in transit - **Advanced audit logs** (7-year retention on Enterprise) - **Advanced access controls** and permissions management - **Dedicated account manager** and support --- ### Additional Configuration Steps After Enablement Once your BAA is signed and the module is active, you'll want to configure: - Role-based access controls (minimum necessary access) - Audit log monitoring - Appropriate data retention policies - Staff training on PHI workflows within the platform --- ### What to Do Next 1. **Contact our Sales team** to discuss Enterprise pricing and initiate the BAA process — you can reach out via enterprise@company.comJudge: The answer correctly identifies the HIPAA compliance module as an Enterprise add-on at $200/month, which matches the ground truth, though it adds numerous unverified details about features and requirements like BAA, SOC 2, encryption specs, and audit logs that are not in the ground truth.","sliced: 7/10, 4062 tok, 9.6s## HIPAA Compliance on TechCorp Platform Thank you for flagging this — handling patient medical data (PHI) requires specific steps before you begin storing it on our platform. Here's what you need to know: ### Requirements for HIPAA Compliance HIPAA compliance on TechCorp requires **all three** of the following to be in place: 1. **Enterprise Plan** — HIPAA support is an Enterprise-tier feature only. 2. **HIPAA Module** — Available as an add-on at **$200/month**, must be explicitly enabled on your account. 3. **Signed Business Associate Agreement (BAA)** — Must be executed with TechCorp before any PHI is stored or processed. ### ⚠️ Important: Do Not Store PHI Until All Three Are Active If any of these elements are missing, **please do not store patient data on the platform yet.** Storing PHI without a signed BAA and the HIPAA module enabled would place your organization in a non-compliant state and could expose you to regulatory risk. ### Once HIPAA Is Enabled, You Should Also Configure: - **Access controls** — Restrict PHI access to authorized personnel only - **Audit logs** — Enable and review audit logging for PHI workflows - **Data retention policies** — Configure appropriate retention periods - **Staff training** — Ensure staff handling PHI are trained on minimum-necessary access principles ### Next Steps | Your Situation | Recommended Action | |---|---| | Already on Enterprise | Contact your account team to add the HIPAA module and initiate BAA signing | | Not yet on Enterprise | Speak with Sales about upgrading your plan | | Unsure of your current plan | Check your admin console or contact support | WouldJudge: The AI answer correctly identifies the HIPAA compliance module as an Enterprise add-on at $200/month, which matches the ground truth, though it adds extra details like BAA requirements and configuration steps that are not in the ground truth but are reasonable and not contradictory."


Full prompt,Sliced prompt
"full: 10/10, 30104 tok, 4.1s## Parental Leave Policy TechCorp offers **16 weeks of fully paid parental leave** for all eligible employees. ### Key Details: - **Duration:** 16 weeks - **Pay:** Fully paid (100% of base salary) - **Eligibility:** All genders - **Qualifying events:** Both birth and adoption This is an internal HR policy applicable to TechCorp employees. If you have specific questions about your situation — such as how parental leave interacts with a performance review cycle or other benefits — I'd recommend reaching out to your HRBP (HR Business Partner) for personalized guidance.Judge: The AI answer accurately captures all ground-truth facts: 16 weeks, fully paid, for birth or adoption, and available to all genders.","sliced: 10/10, 4152 tok, 6.0s## Parental Leave Policy TechCorp offers **16 weeks of fully paid parental leave**, with the following key details: - **Duration:** 16 weeks - **Pay:** 100% paid throughout the leave period - **Eligibility:** Available to **all genders** - **Qualifying Events:** Covers both **birth and adoption** This policy was last updated on **August 1, 2024**, when the language was revised to explicitly reflect inclusion of all genders and adoption as a qualifying event. --- If you have specific questions about how parental leave interacts with your performance review cycle or other benefits, feel free to ask — for example, if your leave overlaps with a review period, you remain eligible for your annual review and your manager should use performance evidence from your active working period.Judge: The AI answer accurately captures all ground-truth facts: 16 weeks, fully paid, for birth or adoption, and available to all genders."


Full prompt,Sliced prompt
"full: 8/10, 30106 tok, 8.1sYes! With 300 users, you qualify for a volume discount on the **Enterprise plan**. Here are the applicable volume discount tiers: | User Range | Additional Discount | |---|---| | 100–250 users | 10% off base price | | **251–500 users** | **15% off base price** | | 500+ users | Custom pricing | Since your 300 users fall in the **251–500 user tier**, you would receive a **15% additional discount** applied to the Enterprise base price. ## Enterprise Base Pricing (before volume discount): - **Monthly:** $299/month - **Annual:** $2,870/year *(already includes 20% annual discount)* ## Next Steps: Given your user count, an **Enterprise Sales team member and Solutions Engineer** would be assigned to work with you directly. You can: 1. **Reach out to** enterprise@company.com to get started 2. An Account Executive will be assigned within 1 business day They can provide your exact pricing with the 15% volume discount applied, discuss any multi-year contract options (which may offer additional savings), and walk through Enterprise features like unlimited users, 1TB storage, dedicated support, and SLA guarantees. Would you like help with anything else, such as a feature comparison or information about the onboarding process?Judge: The AI correctly identifies that 300 users fall in the 251-500 user range and qualify for a 15% additional discount on the Enterprise plan, matching the ground truth, though it adds unverified pricing details and other tiers not in the ground truth.","sliced: 8/10, 3820 tok, 8.9sYes, a volume discount is available for your organization! Here are the details: ## Volume Discount for 300 Users With 300 users, you fall into the **251–500 user tier**, which qualifies for a **15% additional discount** applied to the base Enterprise plan price. ### How the Pricing Breaks Down: | Billing Cycle | Base Price | After 15% Volume Discount | |---|---|---| | Monthly | $299/month | ~$254.15/month | | Annual | $2,870/year | ~$2,439.50/year | > Note: These figures reflect the volume discount on the base Enterprise price. Final pricing may vary based on additional negotiations, multi-year terms, or other factors. --- ### Next Steps Since you have **300 users**, you fall into the **mid-market to Enterprise range**, so here's what to expect: - An **Account Executive will be assigned within 1 business day** to guide you through the process. - You may also be eligible for **additional discounts** through multi-year contracts or competitive pricing, which your AE can discuss. - Enterprise contracts include access to an **MSA, DPA, Net-30 payment terms**, and **Purchase Order acceptance**. To get started, you can reach out directly to **enterprise@company.com** or simply request to be connected with an Account Executive. Would you like help with anything else, such as feature comparisons or contract options?Judge: The AI answer correctly identifies that 300 users falls in the 251-500 user tier and qualifies for a 15% additional discount on the Enterprise plan, which matches the ground truth facts."


Full prompt,Sliced prompt
"full: 8/10, 30104 tok, 11.2s## Importing from Trello Great news — TechCorp has a **native Trello importer** that preserves your board structure. Here's how to get started: ### Steps to Import 1. **Navigate to the importer:** Go to **Settings > Import > Trello** 2. **Authorize your Trello account:** The importer uses OAuth to connect to Trello — no manual exports needed. 3. **Select your boards:** Choose which Trello boards you want to import. 4. **Review and confirm:** The importer will preserve your **boards, lists, cards, and standard fields**. --- ### Before You Begin — A Few Things to Know | Consideration | Detail | |---|---| | **Boards nested in Trello Teams** | Not supported by native import. Export those individually or use a CSV workaround. | | **Recommended approach** | Always **test with one board first** before running a full import. | | **Data format** | If using CSV instead, download our import template at **help.company.com/import-template** | --- ### After Importing - **Review your tasks** to confirm cards mapped correctly. - Check any **custom fields or Power-Up data** from Trello, as non-standard data may not carry over automatically. - Validate assignees — if there are any **email mismatches**, those items may appear unassigned and will need manual cleanup. --- ### Need More Help? - **Starter plan:** Self-serve docs and community forum are available. - **Professional plan:** You can schedule a **live onboarding call (60 min)** for guided migration assistance. - **Enterprise plan:** Your dedicated onboarding manager can walk through aJudge: The AI answer correctly identifies the native Trello import path (Settings > Import > Trello) and confirms that boards are preserved, matching the ground-truth facts, though it adds additional details not verified by the ground truth.","sliced: 8/10, 4102 tok, 9.4s## Importing from Trello Great news — TechCorp supports a **native Trello import**! Here's how to get started: ### Steps to Import 1. Go to **Settings > Import > Trello** 2. Authorize your Trello account when prompted 3. Select the boards you want to import 4. Review the field mapping and confirm 5. Run the import Your **boards will be preserved** through this process. --- ### Important Things to Know Before You Import **Recommended: Test first** - Always run a **test import** on a sample board before doing a full migration - This helps you catch any issues before they affect all your data **Known Limitation — Nested Boards:** - Boards that are **nested inside Trello Teams** are not supported by the native import - For those, you have two options: - Export individual boards separately, then import them - Use the **CSV workaround** (export from Trello as CSV, then use Settings > Import with our CSV template) --- ### After Your Import - Review your imported boards to confirm tasks, assignees, and dates came through correctly - Any boards not included may need to be imported individually or via CSV --- Do you have boards nested in Trello Teams, or are these standard standalone boards? That will help me confirm the best path for your specific situation!Judge: The answer correctly identifies the native Trello import via Settings > Import > Trello and confirms boards are preserved, matching the ground truth, though it adds unverified details about limitations and extra steps not in the ground truth."


Full prompt,Sliced prompt
"full: 8/10, 30109 tok, 9.5s## Your Data Ownership **You retain full and complete ownership of all data you store in the TechCorp platform.** Your data remains yours — we do not claim ownership over it. --- ### What Rights Does TechCorp Have? TechCorp receives only a **limited license** to your data, strictly for the purpose of: - **Operating** the service - **Securing** your data - **Supporting** your use of the platform That's it. The license exists solely to allow us to run the platform on your behalf — not to use your data for any other purpose. --- ### Specific Protections Worth Knowing | Topic | Policy | |---|---| | **AI & Model Training** | Your data is **not** used to train our base AI models | | **Data Export** | You can export your full account data as JSON or CSV **at any time** | | **Right to Erasure** | You can request deletion via privacy@company.com, processed within 30 days | | **Encryption** | Your data is encrypted at rest (AES-256) and in transit (TLS 1.2/1.3) | --- ### For Additional Assurance - **All paid plans** can request a **Data Processing Agreement (DPA)**, which is GDPR Article 28 compliant and includes Standard Contractual Clauses (SCCs). - **Enterprise customers** can also explore **customer-managed encryption keys**, giving you direct control over access to your own data. --- Is there a specific concern — such as GDPR compliance, data residency, or a particular use case — that you'd like me to address in more detail?Judge: The AI answer correctly states that the customer retains full ownership of their data and that the company has only a limited license for service operation, which aligns with the ground-truth facts, though it adds several specific details (encryption specs, DPA, export formats) that are not in the ground truth but do not contradict it.","sliced: 9/10, 4066 tok, 8.7s## Data Ownership **You retain full and complete ownership of all data you store on the TechCorp platform.** This is a firm commitment in our Terms of Service. --- ### TechCorp's Limited Rights TechCorp does receive a **limited license** to your data, but this is strictly scoped: - ✅ **Permitted:** Using the license solely to **operate, secure, and support the service** on your behalf - ❌ **Not permitted:** Using your data for any purpose beyond what is necessary to deliver the service In practical terms, this means we can process your data to run the platform, maintain security, and provide support — but we do not claim ownership, and your data is not used for any other commercial purpose. --- ### Additional Protections to Know | Protection | Detail | |---|---| | **Data Export** | You can export your full account data as JSON or CSV at any time | | **Right to Erasure** | Submit requests to privacy@company.com, processed within 30 days | | **DPA (Paid Plans)** | A Data Processing Agreement formalizing our obligations is available for all paid plans | | **GDPR Compliance** | EU Data Processing Agreement available, including Standard Contractual Clauses | --- If you have specific contractual concerns about data rights — particularly around an Enterprise agreement or legal review — I can route those to our Legal team for formal handling.Judge: The AI answer correctly states that the customer retains full ownership of their data and that the company only holds a limited license scoped to service operation, which aligns precisely with the ground truth facts."
